# Spambase

Objectif : prédire si un e-mail est un **spam (1)** ou non **(0)** à partir de 57 features (fréquences de mots, de caractères, longueurs de suites de majuscules).

Ce notebook couvre :
1. l'import des données,
2. le nettoyage et le prétraitement (valeurs manquantes, doublons, centrage-réduction, ACP),
3. la séparation train / test.

La sélection de features et l'entraînement des modèles viendront ensuite. Toutes les fonctions utilisées sont dans `ml_workflow.py`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

import ml_workflow as mw

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

# Listes de colonnes propres à Spambase
# - colonnes à queue lourde (nombre de majuscules), à passer en log(1 + x)
SPAM_HEAVY_TAILED = ["capital_run_length_average",
                     "capital_run_length_longest",
                     "capital_run_length_total"]
# - features propres aux boîtes mail des créateurs du dataset (HP Labs, George Forman,
#   indicatif 650) : très discriminantes ici, mais elles ne se généraliseraient pas
SPAM_CORPUS_SPECIFIC = ["word_freq_george", "word_freq_650",
                        "word_freq_hp", "word_freq_hpl"]

## 1. Import des données

Le fichier `spambase.data` n'a pas d'en-tête : les noms des 57 colonnes sont lus dans `spambase.names`. La dernière colonne, `spam`, est la cible.

In [ ]:
X, y = mw.load_spambase()
print(X.shape, y.shape)
X.head()

In [ ]:
mw.describe_dataset(X, y)

### Répartition des classes

Environ 39 % de spams : les classes sont **relativement équilibrées**, donc l'accuracy reste une métrique lisible. On stratifiera quand même le split pour conserver cette proportion.

In [ ]:
ax = y.value_counts().rename({0: "non-spam", 1: "spam"}).plot.bar(color=["tab:blue", "tab:red"])
ax.set_title("Répartition des classes")
ax.set_ylabel("Nombre d'e-mails")
plt.xticks(rotation=0)
plt.show()

### Statistiques descriptives

Les 54 features `word_freq_*` et `char_freq_*` sont des pourcentages dans $[0, 100]$. Les 3 features `capital_run_length_*` ont une tout autre échelle et des valeurs extrêmes : la standardisation est donc indispensable pour les modèles basés sur des distances (k-NN, SVM) ou des pénalités (régression logistique régularisée).

In [ ]:
X.describe().T.sort_values("max", ascending=False).head(10)

### Features creuses

La plupart des mots n'apparaissent que dans une petite partie des e-mails : une grande majorité des valeurs sont nulles.

In [ ]:
zero_share = (X == 0).mean().sort_values(ascending=False)
print(f"Part moyenne de zéros par feature : {zero_share.mean():.1%}")
zero_share.head(10)

### Features à queue lourde

Les `capital_run_length_*` sont très asymétriques (médiane faible, maximum de plusieurs milliers). On leur applique $x \mapsto \log(1 + x)$ : cette transformation n'apprend aucun paramètre, donc elle peut être faite avant le split sans fuite de données.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for j, col in enumerate(SPAM_HEAVY_TAILED):
    axes[0, j].hist(X[col], bins=50)
    axes[0, j].set_title(col)
    axes[1, j].hist(np.log1p(X[col]), bins=50, color="tab:green")
    axes[1, j].set_title(f"log(1 + {col.split('_')[-1]})")
axes[0, 0].set_ylabel("brut")
axes[1, 0].set_ylabel("après log(1+x)")
plt.tight_layout()
plt.show()

### Corrélation avec la cible

Première idée des features utiles (corrélation linéaire seulement, la sélection de features sera faite proprement plus tard).

**Attention** : `george`, `650`, `hp` et `hpl` sont des indicateurs de **non-spam propres au corpus** (boîtes mail de HP Labs, d'après la documentation). Elles sont très discriminantes ici mais ne se généraliseraient pas à un autre utilisateur : on comparera les modèles avec et sans ces features (`SPAM_CORPUS_SPECIFIC`).

In [ ]:
corr = X.corrwith(y).sort_values()
top = pd.concat([corr.head(8), corr.tail(12)])
colors = ["tab:orange" if c in SPAM_CORPUS_SPECIFIC else
          ("tab:red" if v > 0 else "tab:blue") for c, v in top.items()]
top.plot.barh(figsize=(8, 7), color=colors)
plt.title("Corrélation avec la cible (orange = spécifique au corpus)")
plt.xlabel("Corrélation de Pearson avec 'spam'")
plt.show()

## 2. Nettoyage

### Valeurs manquantes

La documentation annonce aucune valeur manquante, on le vérifie. La fonction `mw.preprocess` contient tout de même une étape d'imputation (moyenne du train), pour que le workflow reste générique.

In [ ]:
print("Valeurs manquantes :", X.isna().sum().sum())

### Doublons

Des e-mails identiques (features et label) sont présents. On les supprime **avant le split** : sinon une même ligne pourrait se retrouver à la fois dans le train et dans le test, ce qui surestimerait les performances.

In [ ]:
n_before = len(X)
X_clean, y_clean = mw.remove_duplicates(X, y)
print(f"{n_before - len(X_clean)} doublons supprimés -> {len(X_clean)} e-mails")
print("Part de spam après nettoyage :", round(y_clean.mean(), 3))

In [ ]:
X_clean = mw.log_transform(X_clean, SPAM_HEAVY_TAILED)
X_clean[SPAM_HEAVY_TAILED].describe().T

## 3. Séparation train / test

Split **stratifié** 80 / 20 : le test set ne sera utilisé qu'à la toute fin, pour évaluer le modèle retenu. Le choix des hyperparamètres et des features se fera par validation croisée sur le train.

In [ ]:
X_train, X_test, y_train, y_test = mw.split_data(X_clean, y_clean, test_size=0.2,
                                                 random_state=RANDOM_STATE)
print("Train :", X_train.shape, "| part de spam :", round(y_train.mean(), 3))
print("Test  :", X_test.shape, "| part de spam :", round(y_test.mean(), 3))

### Centrage-réduction

Ajusté sur le train uniquement, puis appliqué au test :

$$
z_{ij} = \frac{x_{ij} - \mu_j^{\text{train}}}{\sigma_j^{\text{train}}}
$$

Sur le train, chaque feature a donc une moyenne de 0 et un écart-type de 1 ; sur le test, c'est seulement approximatif, ce qui est normal.

In [ ]:
X_train_s, X_test_s, tr = mw.preprocess(X_train, X_test, impute_strategy="mean", scale=True)
pd.DataFrame({
    "moyenne train": X_train_s.mean().round(3),
    "écart-type train": X_train_s.std(ddof=0).round(3),
    "moyenne test": X_test_s.mean().round(3),
    "écart-type test": X_test_s.std(ddof=0).round(3),
}).head(8)

### ACP

On étudie la variance expliquée par les composantes principales (ACP ajustée sur le train standardisé). Si peu de composantes suffisaient, l'ACP serait une bonne réduction de dimension ; ici, la variance est très étalée.

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE).fit(X_train_s)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

for seuil in (0.8, 0.9, 0.95):
    k = int(np.searchsorted(cum_var, seuil) + 1)
    print(f"{seuil:.0%} de variance expliquée -> {k} composantes sur {X_train_s.shape[1]}")

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(cum_var) + 1), cum_var, marker=".")
plt.axhline(0.9, color="grey", ls="--", lw=1)
plt.xlabel("Nombre de composantes")
plt.ylabel("Variance expliquée cumulée")
plt.title("ACP — variance expliquée cumulée (train)")
plt.show()

In [ ]:
pcs = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_train_s)
plt.figure(figsize=(7, 5))
for label, name, c in [(0, "non-spam", "tab:blue"), (1, "spam", "tab:red")]:
    m = (y_train == label).to_numpy()
    plt.scatter(pcs[m, 0], pcs[m, 1], s=6, alpha=0.4, c=c, label=name)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Projection sur les 2 premières composantes (train)")
plt.legend()
plt.show()

## Jeux de données prêts pour la suite

`mw.prepare_data` refait les étapes 1 à 3 en une ligne. On prépare les variantes à comparer lors de l'entraînement :
- **full** : les 57 features, standardisées ;
- **sans corpus** : sans `george`, `650`, `hp`, `hpl` ;
- **ACP 90 %** : composantes principales expliquant 90 % de la variance.

In [ ]:
datasets = {
    "full": mw.prepare_data("spam", log_cols=SPAM_HEAVY_TAILED,
                            random_state=RANDOM_STATE),
    "acp90": mw.prepare_data("spam", log_cols=SPAM_HEAVY_TAILED,
                             pca_components=0.9, random_state=RANDOM_STATE),
}

# Variante sans les features propres au corpus : on les retire après le split
# (même découpage train/test que 'full', pour comparer à armes égales)
Xtr, Xte, ytr, yte, _ = datasets["full"]
datasets["sans_corpus"] = (Xtr.drop(columns=SPAM_CORPUS_SPECIFIC),
                           Xte.drop(columns=SPAM_CORPUS_SPECIFIC), ytr, yte, None)

for name, (Xtr, Xte, ytr, yte, _) in datasets.items():
    print(f"{name:12s} train {Xtr.shape}  test {Xte.shape}")

**Bilan des étapes 1 à 3**


- 4 601 e-mails, 57 features continues, **aucune valeur manquante**, 391 doublons supprimés.
- Classes à peu près équilibrées (~39 % de spam) → split stratifié, accuracy interprétable.
- Échelles très différentes et features à queue lourde → $\log(1+x)$ sur `capital_run_length_*` puis centrage-réduction.
- L'ACP compresse peu (environ 43 composantes pour 90 % de la variance) : elle sera testée mais ce n'est probablement pas la meilleure réduction.
- Des features propres au corpus (`george`, `650`, `hp`, `hpl`) posent une question de généralisation.

**Prochaines étapes** : sélection de features (information mutuelle, L1, RFE) et comparaison des modèles (régression logistique, Naive Bayes, k-NN, SVM, Random Forest, Gradient Boosting ??), en surveillant particulièrement le **taux de faux positifs**.